# 04-02 文档切分策略（Chunking）

**为什么 Chunking 至关重要**：
- Chunk 太大 → 噪声多，相关信息被稀释，embedding 不精准
- Chunk 太小 → 上下文不完整，LLM 无法理解段落含义
- 最优 Chunk = 语义完整 + 大小适中

**本节目标**：
- 固定大小切分、句子切分、递归切分
- 语义切分（基于 Embedding 的断点检测）
- Chunk 元数据设计

---

In [ ]:
import re, sys
import numpy as np
sys.path.insert(0, "..")

# 模拟 B站广告政策文档
AD_POLICY_DOC = """B站广告投放规范手册 v2.0

第一章 广告基本要求

所有在B站投放的广告内容必须真实、合法、有效，不得含有虚假信息。广告主需对广告内容的真实性负责。广告内容不得违反中华人民共和国广告法及相关法律法规。

广告标题要求：标题字数不超过30字，不得使用极限词（如"最"、"第一"、"绝对"等），不得使用感叹号超过2个。广告正文不超过200字，语言需通俗易懂，与产品实际功能相符。

第二章 广告素材规格

图片广告：支持JPG/PNG格式，文件大小不超过2MB，尺寸为1920×1080像素（16:9）或1080×1080像素（1:1）。图片内文字占比不超过30%，背景不得使用纯黑或纯白。

视频广告：支持MP4/MOV格式，文件大小不超过500MB，时长5-60秒，分辨率不低于720P。视频开始3秒内需出现品牌信息，结尾需有清晰的行动召唤。音频需清晰无杂音。

第三章 投放设置

出价方式：支持CPM（千次展示计费）、CPC（点击计费）、OCPM（智能千次展示计费）三种方式。OCPM是B站推荐的智能出价方式，系统会根据目标自动优化出价。CPM最低出价5元，CPC最低出价0.1元。

定向设置：支持地域、年龄、性别、兴趣、设备、行为等多维度定向。建议新广告主先用宽泛定向积累数据，再逐步收窄。

预算设置：日预算最低100元，总预算最低1000元。系统会在设定预算内智能分配每小时消耗，避免预算在短时间内耗尽。
"""

print(f"文档总字符数: {len(AD_POLICY_DOC)}")
print(f"文档前100字: {AD_POLICY_DOC[:100]}...")

## 1. 固定大小切分

In [ ]:
def fixed_size_chunking(text: str, chunk_size: int = 200, overlap: int = 50) -> list[str]:
    """
    固定字符数切分，带重叠
    overlap 确保跨 chunk 的上下文不丢失
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks_fixed = fixed_size_chunking(AD_POLICY_DOC, chunk_size=200, overlap=50)
print(f"固定大小切分: {len(chunks_fixed)} 个 chunks")
print(f"平均长度: {sum(len(c) for c in chunks_fixed) / len(chunks_fixed):.0f} 字符")
print(f"\n前3个 Chunk:")
for i, chunk in enumerate(chunks_fixed[:3]):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

## 2. 句子级别切分

In [ ]:
def sentence_chunking(text: str, max_sentences: int = 3) -> list[str]:
    """按句子切分，每个 chunk 包含 N 个句子"""
    # 中文句子分隔符
    sentence_endings = r'(?<=[。！？.!?])\s*'
    sentences = re.split(sentence_endings, text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = ''.join(sentences[i:i+max_sentences])
        if chunk:
            chunks.append(chunk)
    return chunks

chunks_sentence = sentence_chunking(AD_POLICY_DOC, max_sentences=2)
print(f"句子切分: {len(chunks_sentence)} 个 chunks")
lengths = [len(c) for c in chunks_sentence]
print(f"长度范围: {min(lengths)}-{max(lengths)} 字符，平均: {sum(lengths)/len(lengths):.0f}")
print(f"\n示例 Chunk:")
print(chunks_sentence[3])

## 3. 递归字符切分（LangChain 风格）

In [ ]:
def recursive_split(
    text: str,
    chunk_size: int = 300,
    overlap: int = 50,
    separators: list[str] = None
) -> list[str]:
    """
    递归字符切分：优先按段落切，再按句子切，最后按字符切
    这是 LangChain RecursiveCharacterTextSplitter 的核心逻辑
    """
    if separators is None:
        separators = ["\n\n", "\n", "。", "！", "？", "，", " ", ""]
    
    # 尝试用当前分隔符切分
    for sep in separators:
        if sep and sep in text:
            splits = text.split(sep)
            splits = [s for s in splits if s.strip()]
            
            # 合并小于 chunk_size 的相邻分块
            chunks = []
            current = ""
            for split in splits:
                if len(current) + len(split) + len(sep) <= chunk_size:
                    current += (sep if current else "") + split
                else:
                    if current:
                        chunks.append(current)
                    if len(split) > chunk_size:
                        # 单个 split 太长，递归处理
                        chunks.extend(recursive_split(split, chunk_size, overlap, separators[1:]))
                    else:
                        current = split
            if current:
                chunks.append(current)
            return chunks
    
    # 最终回退：字符级固定切分
    return fixed_size_chunking(text, chunk_size, overlap)

chunks_recursive = recursive_split(AD_POLICY_DOC, chunk_size=300, overlap=30)
print(f"递归切分: {len(chunks_recursive)} 个 chunks")
lengths = [len(c) for c in chunks_recursive]
print(f"长度范围: {min(lengths)}-{max(lengths)} 字符，平均: {sum(lengths)/len(lengths):.0f}")
print(f"\n示例 Chunk（语义更完整）:")
print(chunks_recursive[1])

## 4. 语义切分（基于 Embedding 断点检测）

In [ ]:
def semantic_chunking(text: str, threshold: float = 0.3) -> list[str]:
    """
    语义切分：当相邻句子的语义相似度低于阈值时，切分
    原理：相似度骤降 = 话题切换 = 切分点
    """
    # 先按句子切分
    sentences = re.split(r'(?<=[。！？\n])\s*', text)
    sentences = [s.strip() for s in sentences if s.strip() and len(s) > 5]
    
    if len(sentences) < 3:
        return [text]
    
    # 计算每个句子的 embedding
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
        embeddings = model.encode(sentences, normalize_embeddings=True)
    except ImportError:
        # Mock：相邻句子相似度高，段落间低
        embeddings = []
        for i, sent in enumerate(sentences):
            rng = np.random.default_rng(hash(sent) % (2**32))
            base = rng.standard_normal(64)
            # 同段落内的句子用相同基础向量加小噪声
            base /= np.linalg.norm(base)
            embeddings.append(base)
        embeddings = np.array(embeddings)
    
    # 计算相邻句子相似度
    similarities = []
    for i in range(len(embeddings) - 1):
        sim = float(np.dot(embeddings[i], embeddings[i+1]))
        similarities.append(sim)
    
    # 找切分点（相似度低于阈值）
    breakpoints = [i+1 for i, s in enumerate(similarities) if s < threshold]
    
    # 按切分点组合句子
    chunks = []
    start = 0
    for bp in breakpoints + [len(sentences)]:
        chunk = ''.join(sentences[start:bp])
        if chunk:
            chunks.append(chunk)
        start = bp
    
    return chunks

chunks_semantic = semantic_chunking(AD_POLICY_DOC, threshold=0.5)
print(f"语义切分: {len(chunks_semantic)} 个 chunks")
print(f"\n各 Chunk 长度: {[len(c) for c in chunks_semantic]}")

## 5. Chunk 元数据设计

In [ ]:
from datetime import datetime
from dataclasses import dataclass, field

@dataclass
class DocumentChunk:
    """带有丰富元数据的文档块"""
    content: str
    chunk_id: str
    doc_id: str
    chunk_index: int
    total_chunks: int
    source: str = ""
    doc_title: str = ""
    char_count: int = field(init=False)
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    
    def __post_init__(self):
        self.char_count = len(self.content)
    
    def to_dict(self) -> dict:
        return {
            "content": self.content,
            "chunk_id": self.chunk_id,
            "doc_id": self.doc_id,
            "chunk_index": self.chunk_index,
            "total_chunks": self.total_chunks,
            "source": self.source,
            "doc_title": self.doc_title,
            "char_count": self.char_count,
            "created_at": self.created_at,
        }

def create_chunks_with_metadata(
    text: str, doc_id: str, source: str, title: str,
    chunk_size: int = 300
) -> list[DocumentChunk]:
    raw_chunks = recursive_split(text, chunk_size=chunk_size)
    return [
        DocumentChunk(
            content=chunk,
            chunk_id=f"{doc_id}_chunk_{i:03d}",
            doc_id=doc_id,
            chunk_index=i,
            total_chunks=len(raw_chunks),
            source=source,
            doc_title=title,
        )
        for i, chunk in enumerate(raw_chunks)
    ]

chunks = create_chunks_with_metadata(
    AD_POLICY_DOC,
    doc_id="bilibili_ad_policy_v2",
    source="内部文档",
    title="B站广告投放规范手册 v2.0"
)

print(f"带元数据的 chunks: {len(chunks)} 个")
import json
print("\n示例 Chunk 元数据:")
sample = chunks[1].to_dict()
sample["content"] = sample["content"][:50] + "..."
print(json.dumps(sample, ensure_ascii=False, indent=2))

## 切分策略对比

| 策略 | Chunk 数 | 平均大小 | 优点 | 缺点 | 适用场景 |
|------|---------|---------|------|------|----------|
| 固定大小 | 多 | 均匀 | 简单、可控 | 可能切断语义 | 非结构化文本 |
| 句子级 | 中 | 不均匀 | 语义完整 | chunk可能过小 | 对话、FAQ |
| 递归切分 | 中 | 较均匀 | 尊重文档结构 | 参数敏感 | **大多数场景首选** |
| 语义切分 | 少 | 变化大 | 最佳语义边界 | 计算成本高 | 长文档、高精度需求 |

## 面试速记

| 问题 | 要点 |
|------|------|
| 最优 chunk size | 经验值：200-500 tokens；需根据 Embedding 模型最大长度和任务调整 |
| overlap 的作用 | 防止跨 chunk 的关键信息丢失，建议 10-20% 重叠 |
| 元数据的重要性 | 检索后可按 source/date 过滤，提供来源引用，支持混合检索 |
| 语义切分的代价 | 每个句子都要 embed，计算成本高，适合高价值文档 |

**下一节**: `03_retrieval_ranking.ipynb`